In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Installing  python 3.10 as fairseq doesnt run otherwise.

In [ ]:
# Install Python 3.10
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils python3.10-dev -y

# Update alternatives
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 2
!sudo update-alternatives --config python3  # choose Python 3.10

# Reinstall pip for Python 3.10
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10


Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,287 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [5,988 kB]
Get:

Installing all the dependencies

In [ ]:
import subprocess
import os

print("Installing dependencies...")
# Downgrade pip to avoid omegaconf metadata issue
subprocess.run("pip install 'pip<24.1'", shell=True, check=True)

# Clone fairseq repository if not already done
if not os.path.exists("/content/fairseq"):
    print("Cloning fairseq repository...")
    subprocess.run("git clone https://github.com/facebookresearch/fairseq /content/fairseq", shell=True, check=True)

# Install fairseq with compatible dependencies
print("Installing fairseq from source...")
subprocess.run(
    "cd /content/fairseq && "
    "pip install --editable . "
    "omegaconf==2.0.6 hydra-core==1.0.7",
    shell=True,
    check=True
)

# Install other dependencies
subprocess.run("pip install sentencepiece sacrebleu", shell=True, check=True)

print("Dependencies installed.")

Installing dependencies...
Cloning fairseq repository...
Installing fairseq from source...
Dependencies installed.


In [ ]:
!python3.10 -m pip install unbabel-comet
!python3.10 -c "from comet import download_model, load_from_checkpoint;"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Defining utility functions

In [ ]:
import os
import subprocess
import shutil

# Function to run shell commands
def run_shell_command(command, capture_output=True):
    try:
        result = subprocess.run(command, shell=True, check=True, text=True, capture_output=capture_output)
        if capture_output:
            print(f"Command output: {result.stdout}")
            if result.stderr:
                print(f"Command error: {result.stderr}")
        print(f"Command succeeded: {command}")
        return result
    except subprocess.CalledProcessError as e:
        print(f"Error in command {command}: {e.stderr}")
        raise

# Function to prepend language tag to a file
def prepend_tag(file_path, tag, output_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = [f"{tag} {line.strip()}\n" for line in f]
        with open(output_path, 'w', encoding='utf-8') as f:
            f.writelines(lines)
        print(f"Prepended {tag} to {file_path} -> {output_path}")
    except Exception as e:
        raise RuntimeError(f"Failed to prepend tag to {file_path}: {str(e)}")
print("Utility functions defined.")

Utility functions defined.


In [ ]:
!python3.10 -m pip install sacrebleu
!python3.10 -c "import sacrebleu; print('SacreBLEU version:', sacrebleu.__version__)"


DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
SacreBLEU version: 2.5.1


Verifying input data

In [ ]:
import os

In [ ]:
# Input files in /content/drive/MyDrive/MT_data, outputs in /content/MT_output
input_dir = "/content/drive/MyDrive/MT_data"
output_dir = "/content/MT_output2"

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)
train_src_input = os.path.join(input_dir,"ne-train.txt")  # Nepali
train_tgt_input = os.path.join(input_dir,"tmg-train.txt")  # Tamang
test_src = os.path.join(input_dir, "ne-test.txt")  # Same for test
test_tgt = os.path.join(input_dir, "tmg-test.txt")

# Verify input files exist
for f in [train_src_input, train_tgt_input, test_src, test_tgt]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Input file missing: {f}")
print("All input files verified.")

All input files verified.


Preparing Training Data

In [ ]:
import os, shutil

print("Preparing training data...")

def clean_lines(path):
    with open(path, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]
    return lines

# Clean + load both original files
src_lines = clean_lines(train_src_input)  # Nepali
tgt_lines = clean_lines(train_tgt_input)  # Tamang

# Make sure same number of lines
min_len = min(len(src_lines), len(tgt_lines))
if len(src_lines) != len(tgt_lines):
    print(f"⚠️ Line mismatch fixed: src={len(src_lines)}, tgt={len(tgt_lines)} → trimming to {min_len}")
src_lines, tgt_lines = src_lines[:min_len], tgt_lines[:min_len]

# Write forward: ne → taj
train_forward_src = os.path.join(output_dir, "train_forward.src")
train_forward_tgt = os.path.join(output_dir, "train_forward.tgt")
with open(train_forward_src, 'w', encoding='utf-8') as f:
    for line in src_lines:
        f.write(f"__taj__ {line}\n")
with open(train_forward_tgt, 'w', encoding='utf-8') as f:
    for line in tgt_lines:
        f.write(f"{line}\n")

# Write reverse: taj → ne
train_reverse_src = os.path.join(output_dir, "train_reverse.src")
train_reverse_tgt = os.path.join(output_dir, "train_reverse.tgt")
with open(train_reverse_src, 'w', encoding='utf-8') as f:
    for line in tgt_lines:
        f.write(f"__ne__ {line}\n")
with open(train_reverse_tgt, 'w', encoding='utf-8') as f:
    for line in src_lines:
        f.write(f"{line}\n")

# Combine both into bidirectional files
train_bi_src = os.path.join(output_dir, "train_bi.src")
train_bi_tgt = os.path.join(output_dir, "train_bi.tgt")

with open(train_bi_src, 'w', encoding='utf-8') as out_src, \
     open(train_bi_tgt, 'w', encoding='utf-8') as out_tgt:
    for s, t in zip(src_lines, tgt_lines):
        out_src.write(f"__taj__ {s}\n")
        out_tgt.write(f"{t}\n")
    for s, t in zip(tgt_lines, src_lines):
        out_src.write(f"__ne__ {s}\n")
        out_tgt.write(f"{t}\n")

print("✅ Training data prepared and verified for both directions.")


Preparing training data...
✅ Training data prepared and verified for both directions.


Checking the alignment of the data. Can skip running this

In [ ]:
# This is THE MOST IMPORTANT check - verify the concatenation preserved order
with open("/content/MT_output2/train_bi.src", 'r', encoding='utf-8') as f:
    bi_src_lines = f.readlines()

with open("/content/MT_output2/train_bi.tgt", 'r', encoding='utf-8') as f:
    bi_tgt_lines = f.readlines()

# Check first few forward pairs (ne→taj)
print("=== First 10 FORWARD pairs (ne→taj) ===")
for i in range(10):
    print(f"Source {i+1}: {bi_src_lines[i].strip()[:150]}")
    print(f"Target {i+1}: {bi_tgt_lines[i].strip()[:150]}")
    print()

# Check first few reverse pairs (should be after all forward pairs)
with open("/content/MT_output2/train_forward.src", 'r', encoding='utf-8') as f:
    forward_count = len(f.readlines())

print(f"\n=== First 10 REVERSE pairs (taj→ne) starting at line {forward_count} ===")
for i in range(forward_count, forward_count + 10):
    print(f"Source {i+1}: {bi_src_lines[i].strip()[:150]}")
    print(f"Target {i+1}: {bi_tgt_lines[i].strip()[:150]}")
    print()


=== First 10 FORWARD pairs (ne→taj) ===
Source 1: __taj__ सबै विश्वासमा चलेको छ।
Target 1: मोकोन विश्वासरि बोल्बा मुला।

Source 2: __taj__ आजभोलि काँक्रो खानुभएको छ?
Target 2: तिनिन्हाङगर लाङाइ सोल्बा मुला ?

Source 3: __taj__ यसप्रति सबैको ध्यान जानुपर्छ।
Target 3: चुछाप मोकोनला ध्यान नितोला।

Source 4: __taj__ उनी त्यहाँ सहभागी थिए।
Target 4: थे थेरि सहभागी मुला।

Source 5: __taj__ उनले आफैँ तालिम लिए।
Target 5: थेसे राङनोन तालिम किन्जि।

Source 6: __taj__ हामीसँग डीएपी मल छैन।
Target 6: ङानितेन डीएपी सार आरे।

Source 7: __taj__ अब हामी कहाँ जाने?
Target 7: दारेम ङानि खानाङ निबा ?

Source 8: __taj__ अहिलेसम्म मल पाइएको छैन।
Target 8: दान्देदोःना सार याङ्बा आरे।

Source 9: __taj__ सक्खरलाई बजारको चिन्ता छैन।
Target 9: सक्खरदा बजारला लोतु आरे।

Source 10: __taj__ आफ्नो जग्गा जमिन थिएन।
Target 10: ह्राङला सा आरेबा।


=== First 10 REVERSE pairs (taj→ne) starting at line 15001 ===
Source 15002: __ne__ मोकोन विश्वासरि बोल्बा मुला।
Target 15002: सबै विश्वासमा चलेको छ।

Source 15003: __ne__ 

Preparing test data

In [ ]:
print("Preparing test data...")
# ne->taj
test_ne_taj_src = os.path.join(output_dir, "test_ne_taj.src")
test_ne_taj_tgt = os.path.join(output_dir, "test_ne_taj.tgt")
prepend_tag(test_src, "__taj__", test_ne_taj_src)
shutil.copy(test_tgt, test_ne_taj_tgt)

# taj->ne
test_taj_ne_src = os.path.join(output_dir, "test_taj_ne.src")
test_taj_ne_tgt = os.path.join(output_dir, "test_taj_ne.tgt")
prepend_tag(test_tgt, "__ne__", test_taj_ne_src)
shutil.copy(test_src, test_taj_ne_tgt)
print("Test data prepared.")

Preparing test data...
Prepended __taj__ to /content/drive/MyDrive/MT_data/ne-test.txt -> /content/MT_output2/test_ne_taj.src
Prepended __ne__ to /content/drive/MyDrive/MT_data/tmg-test.txt -> /content/MT_output2/test_taj_ne.src
Test data prepared.


Training sentencepiece model

In [ ]:
import sentencepiece as spm
from collections import Counter
import os

print("Training SentencePiece model...")
combined_file = os.path.join(output_dir, "train_combined.txt")
with open(combined_file, 'w', encoding='utf-8') as f:
    with open(train_bi_src, 'r', encoding='utf-8') as fs:
        f.writelines(fs.readlines())
    with open(train_bi_tgt, 'r', encoding='utf-8') as ft:
        f.writelines(ft.readlines())

spm_model_prefix = os.path.join(output_dir, "spm")
spm.SentencePieceTrainer.Train(
    input=combined_file,
    model_prefix=spm_model_prefix,
    vocab_size=8000,
    character_coverage=1.0,
    model_type="unigram",
    user_defined_symbols="__ne__,__taj__",
    bos_id=0,
    pad_id=1,
    eos_id=2,
    unk_id=3
)

# Load the trained model
sp = spm.SentencePieceProcessor()
sp.load(f"{spm_model_prefix}.model")

# Count actual token frequencies from both source and target files
token_counts = Counter()

def count_tokens_in_file(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            tokens = sp.encode_as_pieces(line.strip())
            token_counts.update(tokens)

# Count tokens in both source and target files
count_tokens_in_file(train_bi_src)
count_tokens_in_file(train_bi_tgt)

# Convert vocab to Fairseq dict with actual frequencies
spm_vocab = f"{spm_model_prefix}.vocab"
dict_txt = os.path.join(output_dir, "dict.txt")

# Read all vocab lines (including special tokens)
with open(spm_vocab, 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open(dict_txt, 'w', encoding='utf-8') as f:
    for line in lines:
        parts = line.split('\t')
        if len(parts) >= 1:
            token = parts[0]
            # Use actual count if available, otherwise use 1
            count = token_counts.get(token, 1)
            f.write(f"{token} {count}\n")

print("SentencePiece model trained with actual frequencies.")

Training SentencePiece model...
SentencePiece model trained with actual frequencies.


Encoding files with sentencepiece model

In [ ]:
print("Encoding files with SentencePiece...")
sp = spm.SentencePieceProcessor(model_file=f"{spm_model_prefix}.model")

def encode_file(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f]
    encoded = [' '.join(sp.encode_as_pieces(line)) + '\n' for line in lines]
    with open(output_path, 'w', encoding='utf-8') as f:
        f.writelines(encoded)


# Encode bi files
train_bi_spm_src = os.path.join(output_dir, "train_bi.spm.src")
train_bi_spm_tgt = os.path.join(output_dir, "train_bi.spm.tgt")
encode_file(train_bi_src, train_bi_spm_src)
encode_file(train_bi_tgt, train_bi_spm_tgt)

# Encode test
test_ne_taj_spm_src = os.path.join(output_dir, "test_ne_taj.spm.src")
test_ne_taj_spm_tgt = os.path.join(output_dir, "test_ne_taj.spm.tgt")
encode_file(test_ne_taj_src, test_ne_taj_spm_src)
encode_file(test_ne_taj_tgt, test_ne_taj_spm_tgt)

test_taj_ne_spm_src = os.path.join(output_dir, "test_taj_ne.spm.src")
test_taj_ne_spm_tgt = os.path.join(output_dir, "test_taj_ne.spm.tgt")
encode_file(test_taj_ne_src, test_taj_ne_spm_src)
encode_file(test_taj_ne_tgt, test_taj_ne_spm_tgt)
print("Files encoded.")

Encoding files with SentencePiece...
Files encoded.


Inspecting and checking the prepared data for understanding. Can skip running this

In [ ]:
src = "/content/MT_output2/train_bi.spm.src"
tgt = "/content/MT_output2/train_bi.spm.tgt"

with open(src, encoding="utf-8") as f:
    src_lines = [l for l in f if l.strip()]
with open(tgt, encoding="utf-8") as f:
    tgt_lines = [l for l in f if l.strip()]

print("Non-empty source lines:", len(src_lines))
print("Non-empty target lines:", len(tgt_lines))

if len(src_lines) != len(tgt_lines):
    print("⚠️ Difference at line:", min(len(src_lines), len(tgt_lines)) + 1)
    if len(src_lines) > len(tgt_lines):
        print("Extra in source:", src_lines[-1][:100])
    else:
        print("Extra in target:", tgt_lines[-1][:100])


Non-empty source lines: 30002
Non-empty target lines: 30002


Binarizing data

In [ ]:
print("Binarizing data...")
data_bin_dir = os.path.join(output_dir, "data-bin")
os.makedirs(data_bin_dir, exist_ok=True)

preprocess_cmd = (
    f"fairseq-preprocess "
    f"--source-lang src --target-lang tgt "
    f"--trainpref {os.path.splitext(train_bi_spm_src)[0]} "
    f"--destdir {data_bin_dir} "
    f"--joined-dictionary "
    f"--srcdict {dict_txt} "
    f"--workers 8"
)
run_shell_command(preprocess_cmd)
# For Tamang -> Nepali test
data_bin_taj_ne = os.path.join(output_dir, "data-bin_taj_ne")
os.makedirs(data_bin_taj_ne, exist_ok=True)
preprocess_cmd_taj_ne = (
    f"fairseq-preprocess "
    f"--source-lang src --target-lang tgt "
    f"--testpref {os.path.splitext(test_taj_ne_spm_src)[0]} "
    f"--destdir {data_bin_taj_ne} "
    f"--joined-dictionary "
    f"--srcdict {dict_txt} "
    f"--workers 8"
)
run_shell_command(preprocess_cmd_taj_ne)

# For Nepali -> Tamang test (if needed separately)
data_bin_ne_taj = os.path.join(output_dir, "data-bin_ne_taj")
os.makedirs(data_bin_ne_taj, exist_ok=True)
preprocess_cmd_ne_taj = (
    f"fairseq-preprocess "
    f"--source-lang src --target-lang tgt "
    f"--testpref {os.path.splitext(test_ne_taj_spm_src)[0]} "
    f"--destdir {data_bin_ne_taj} "
    f"--joined-dictionary "
    f"--srcdict {dict_txt} "
    f"--workers 8"
)
run_shell_command(preprocess_cmd_ne_taj)
print("Binarized")

Binarizing data...
Command output: 
Command error: INFO:fairseq.tasks.text_to_speech:Please install tensorboardX: pip install tensorboardX
INFO:fairseq_cli.preprocess:Namespace(no_progress_bar=False, log_interval=100, log_format=None, log_file=None, aim_repo=None, aim_run_hash=None, tensorboard_logdir=None, wandb_project=None, azureml_logging=False, seed=1, cpu=False, tpu=False, bf16=False, memory_efficient_bf16=False, fp16=False, memory_efficient_fp16=False, fp16_no_flatten_grads=False, fp16_init_scale=128, fp16_scale_window=None, fp16_scale_tolerance=0.0, on_cpu_convert_precision=False, min_loss_scale=0.0001, threshold_loss_scale=None, amp=False, amp_batch_retries=2, amp_init_scale=128, amp_scale_window=None, user_dir=None, empty_cache_freq=0, all_gather_list_size=16384, model_parallel_size=1, quantization_config_path=None, profile=False, reset_logging=False, suppress_crashes=False, use_plasma_view=False, plasma_path='/tmp/plasma', criterion='cross_entropy', tokenizer=None, bpe=None,

Training the model

In [ ]:

!fairseq-train /content/MT_output2/data-bin \
  --arch transformer_iwslt_de_en \
  --encoder-layers 5 --decoder-layers 5 \
  --encoder-embed-dim 512 --decoder-embed-dim 512 \
  --encoder-ffn-embed-dim 2048 --decoder-ffn-embed-dim 2048 \
  --encoder-attention-heads 8 --decoder-attention-heads 8 \
  --share-all-embeddings \
  --optimizer adam --adam-betas '(0.9, 0.98)' --clip-norm 0.0 \
  --lr 5e-4 --lr-scheduler inverse_sqrt --warmup-updates 4000 --warmup-init-lr 1e-07 \
  --dropout 0.3 --weight-decay 1e-4 \
  --criterion label_smoothed_cross_entropy --label-smoothing 0.2 \
  --max-tokens 4096 \
  --disable-validation \
  --fp16 \
  --save-dir /content/MT_output4/checkpoints \
  --max-epoch 60 \
  --patience 5


2025-10-19 05:55:44 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
2025-10-19 05:55:46 | INFO | fairseq_cli.train | {'_name': None, 'common': {'_name': None, 'no_progress_bar': False, 'log_interval': 100, 'log_format': None, 'log_file': None, 'aim_repo': None, 'aim_run_hash': None, 'tensorboard_logdir': None, 'wandb_project': None, 'azureml_logging': False, 'seed': 1, 'cpu': False, 'tpu': False, 'bf16': False, 'memory_efficient_bf16': False, 'fp16': True, 'memory_efficient_fp16': False, 'fp16_no_flatten_grads': False, 'fp16_init_scale': 128, 'fp16_scale_window': None, 'fp16_scale_tolerance': 0.0, 'on_cpu_convert_precision': False, 'min_loss_scale': 0.0001, 'threshold_loss_scale': None, 'amp': False, 'amp_batch_retries': 2, 'amp_init_scale': 128, 'amp_scale_window': None, 'user_dir': None, 'empty_cache_freq': 0, 'all_gather_list_size': 16384, 'model_parallel_size': 1, 'quantization_config_path': None, 'profile': False, 'reset_logging': Fals

Evaluating the model

In [ ]:
import os
import subprocess
import torch


output_dir = "/content/MT_output4"
data_bin_ne_taj = "/content/MT_output2/data-bin_ne_taj"
data_bin_taj_ne = "/content/MT_output2/data-bin_taj_ne"

checkpoints_dir = os.path.join(output_dir, "checkpoints")

def run_shell_command(command, capture_output=True):
    try:
        result = subprocess.run(command, shell=True, check=True, text=True, capture_output=capture_output)
        if capture_output:
            print(f"Command output: {result.stdout}")
            if result.stderr:
                print(f"Command error: {result.stderr}")
        print(f"Command succeeded: {command}")
        return result
    except subprocess.CalledProcessError as e:
        print(f"Error in command {command}: {e.stderr}")
        raise

import argparse
torch.serialization.add_safe_globals([argparse.Namespace])

print("Evaluating model...")

# ------------------ Generate translations ------------------
# ne->taj
gen_out_ne_taj = os.path.join(output_dir, "gen_ne_taj.out")
generate_cmd_ne_taj = (
    f"fairseq-generate {data_bin_ne_taj} "
    f"--path {os.path.join(checkpoints_dir, 'checkpoint60.pt')} "
    f"--source-lang src --target-lang tgt "
    f"--batch-size 128 --beam 5 "
    f"--scoring bleu "
    f"--remove-bpe sentencepiece "
    f"--gen-subset test > {gen_out_ne_taj}"
)
run_shell_command(generate_cmd_ne_taj)

gen_hyp_ne_taj = os.path.join(output_dir, "gen_ne_taj.hyp")
gen_ref_ne_taj = os.path.join(output_dir, "gen_ne_taj.ref")
os.system(f"grep ^H {gen_out_ne_taj} | cut -f3- > {gen_hyp_ne_taj}")
os.system(f"grep ^T {gen_out_ne_taj} | cut -f2- > {gen_ref_ne_taj}")

# taj->ne
gen_out_taj_ne = os.path.join(output_dir, "gen_taj_ne.out")
generate_cmd_taj_ne = (
    f"fairseq-generate {data_bin_taj_ne} "
    f"--path {os.path.join(checkpoints_dir, 'checkpoint60.pt')} "
    f"--source-lang src --target-lang tgt "
    f"--batch-size 128 --beam 5 "
    f"--remove-bpe sentencepiece "
    f"--scoring bleu "
    f"--gen-subset test > {gen_out_taj_ne}"
)
run_shell_command(generate_cmd_taj_ne)

gen_hyp_taj_ne = os.path.join(output_dir, "gen_taj_ne.hyp")
gen_ref_taj_ne = os.path.join(output_dir, "gen_taj_ne.ref")
os.system(f"grep ^H {gen_out_taj_ne} | cut -f3- > {gen_hyp_taj_ne}")
os.system(f"grep ^T {gen_out_taj_ne} | cut -f2- > {gen_ref_taj_ne}")

gen_src_ne_taj = os.path.join(output_dir, "gen_ne_taj.src")
os.system(f"grep ^S {gen_out_ne_taj} | cut -f2- > {gen_src_ne_taj}")

gen_src_taj_ne = os.path.join(output_dir, "gen_taj_ne.src")
os.system(f"grep ^S {gen_out_taj_ne} | cut -f2- > {gen_src_taj_ne}")



Evaluating model...
Command output: 
Command error: INFO:fairseq.tasks.text_to_speech:Please install tensorboardX: pip install tensorboardX
DEBUG:hydra.core.utils:Setting JobRuntime:name=UNKNOWN_NAME
DEBUG:hydra.core.utils:Setting JobRuntime:name=utils
INFO:fairseq_cli.generate:{'_name': None, 'common': {'_name': None, 'no_progress_bar': False, 'log_interval': 100, 'log_format': None, 'log_file': None, 'aim_repo': None, 'aim_run_hash': None, 'tensorboard_logdir': None, 'wandb_project': None, 'azureml_logging': False, 'seed': 1, 'cpu': False, 'tpu': False, 'bf16': False, 'memory_efficient_bf16': False, 'fp16': False, 'memory_efficient_fp16': False, 'fp16_no_flatten_grads': False, 'fp16_init_scale': 128, 'fp16_scale_window': None, 'fp16_scale_tolerance': 0.0, 'on_cpu_convert_precision': False, 'min_loss_scale': 0.0001, 'threshold_loss_scale': None, 'amp': False, 'amp_batch_retries': 2, 'amp_init_scale': 128, 'amp_scale_window': None, 'user_dir': None, 'empty_cache_freq': 0, 'all_gather_l

0

Extracting scores

In [ ]:
cat /content/MT_output4/gen_ne_taj.out | grep BLEU

Generate test with beam=5: BLEU4 = 37.71, 65.5/44.4/31.3/22.5 (BP=0.998, ratio=0.998, syslen=49605, reflen=49725)


In [ ]:
cat /content/MT_output4/gen_taj_ne.out | grep BLEU

Generate test with beam=5: BLEU4 = 38.01, 64.8/45.0/32.0/23.3 (BP=0.989, ratio=0.990, syslen=49449, reflen=49971)


Calculating meteor score

In [ ]:
from nltk.translate import meteor_score
import nltk

# Download WordNet (needed for METEOR)
nltk.download('wordnet')

# Paths to your hypothesis and reference files
hyp_path = '/content/MT_output4/gen_ne_taj.hyp'
ref_path = '/content/MT_output4/gen_ne_taj.ref'

# Read lines (strip whitespace)
with open(hyp_path, 'r', encoding='utf-8') as hyp_file, open(ref_path, 'r', encoding='utf-8') as ref_file:
    hypotheses = [line.strip() for line in hyp_file.readlines()]
    references = [line.strip() for line in ref_file.readlines()]

# Tokenize the sentences (split by whitespace)
hypotheses_tokenized = [hyp.split() for hyp in hypotheses]
references_tokenized = [ref.split() for ref in references]  # Single reference per hypothesis

# Compute sentence-level METEOR scores
sentence_scores = []
for ref, hyp in zip(references_tokenized, hypotheses_tokenized):
    score = meteor_score.single_meteor_score(ref, hyp)
    sentence_scores.append(score)

# Compute corpus-level METEOR score (average of sentence scores)
corpus_meteor = sum(sentence_scores) / len(sentence_scores) if sentence_scores else 0.0

# Print results
for i, score in enumerate(sentence_scores):
    print(f"Sentence {i+1} METEOR: {score:.4f}")

print(f"\nCorpus METEOR score: {corpus_meteor:.4f}")

[nltk_data] Downloading package wordnet to /root/nltk_data...


Sentence 1 METEOR: 0.9815
Sentence 2 METEOR: 0.2500
Sentence 3 METEOR: 0.9375
Sentence 4 METEOR: 0.2500
Sentence 5 METEOR: 0.9815
Sentence 6 METEOR: 0.9815
Sentence 7 METEOR: 0.2500
Sentence 8 METEOR: 0.0000
Sentence 9 METEOR: 0.6250
Sentence 10 METEOR: 0.9815
Sentence 11 METEOR: 0.9375
Sentence 12 METEOR: 0.9375
Sentence 13 METEOR: 0.9815
Sentence 14 METEOR: 0.6048
Sentence 15 METEOR: 0.9815
Sentence 16 METEOR: 0.1667
Sentence 17 METEOR: 0.3333
Sentence 18 METEOR: 0.9815
Sentence 19 METEOR: 0.9815
Sentence 20 METEOR: 0.6250
Sentence 21 METEOR: 0.1667
Sentence 22 METEOR: 0.0000
Sentence 23 METEOR: 0.2500
Sentence 24 METEOR: 0.2500
Sentence 25 METEOR: 0.3333
Sentence 26 METEOR: 0.3333
Sentence 27 METEOR: 0.6250
Sentence 28 METEOR: 0.9815
Sentence 29 METEOR: 0.2632
Sentence 30 METEOR: 0.1724
Sentence 31 METEOR: 0.9815
Sentence 32 METEOR: 0.9815
Sentence 33 METEOR: 0.6250
Sentence 34 METEOR: 0.3333
Sentence 35 METEOR: 0.9815
Sentence 36 METEOR: 0.9815
Sentence 37 METEOR: 0.2500
Sentence 3